In [0]:
!pip install kaggle

In [0]:
import os

os.environ["KAGGLE_USERNAME"] = dbutils.secrets.get("kaggle-scope", "kaggle_username")
os.environ["KAGGLE_KEY"] = dbutils.secrets.get("kaggle-scope", "kaggle_key")

print("KAGGLE_USERNAME set:", bool(os.environ.get("KAGGLE_USERNAME")))
print("KAGGLE_KEY set:", bool(os.environ.get("KAGGLE_KEY")))


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data

kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

In [0]:

%restart_python

In [0]:
df_n = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

In [0]:
df_o = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")

In [0]:
df_oct = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("c")
)

df_nov = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")
)


In [0]:
print(f"October 2019 - Total Events: {df_oct.count():,}")
print(f"Noveber 2019 - Total Events: {df_nov.count():,}")

In [0]:
print("\n" + "="*60)
print("SAMPLE DATA (First 5 rows):")
print("="*60)
df_nov.show(5, truncate=False)

In [0]:

(df_nov.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("events_table")
)


In [0]:
%sql

SELECT COUNT(*) FROM events_table;
DESCRIBE HISTORY events_table;


In [0]:
%sql

DESCRIBE EXTENDED events_table;



In [0]:
%sql

SELECT * FROM events_table VERSION AS OF 0;
-- or
SELECT * FROM events_table TIMESTAMP AS OF '2026-01-13';



In [0]:
%sql
OPTIMIZE events_table ZORDER BY (event_time, user_id, event_type);
VACUUM events_table RETAIN 168 HOURS;


In [0]:
from pyspark.sql import functions as F

df_oct_clean = (df_oct
    .select(
        "event_time","event_type","product_id","category_id","category_code",
        "brand","price","user_id","user_session"
    )
    .withColumn("event_time", F.col("event_time").cast("timestamp"))
    .withColumn("price", F.col("price").cast("double"))
    .dropDuplicates(["user_session", "event_time", "product_id"])
)



In [0]:
from delta.tables import DeltaTable

events_delta = DeltaTable.forName(spark, "events_table")

(events_delta.alias("t")
 .merge(
     df_oct_clean.alias("s"),
     """
     t.user_session = s.user_session
     AND t.event_time = s.event_time
     AND t.product_id = s.product_id
     """
 )
 .whenMatchedUpdate(set={
     "event_type": "s.event_type",
     "category_id": "s.category_id",
     "category_code": "s.category_code",
     "brand": "s.brand",
     "price": "s.price",
     "user_id": "s.user_id"
 })
 .whenNotMatchedInsertAll()
 .execute()
)


In [0]:

spark.table("events_table").count()


*DAY 6 * 

In [0]:

from pyspark.sql import functions as F

BASE_PATH = "/Volumes/workspace/ecommerce/ecommerce_data"
RAW_PATH  = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv"  # adjust to your file location

BRONZE_PATH = f"{BASE_PATH}/bronze/events_raw"

raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(RAW_PATH)
)

bronze = (
    raw
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingestion_date", F.to_date("ingestion_ts"))
)

(bronze.write
 .format("delta")
 .mode("append")
 .partitionBy("ingestion_date")
 .save(BRONZE_PATH))



In [0]:
bronze.select("_metadata.*").display()


In [0]:
SILVER_PATH = f"{BASE_PATH}/silver/events"

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

silver_stage = (
    bronze_df
    # Parse/standardize types
    .withColumn("event_time", F.to_timestamp("event_time"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn("event_date", F.to_date("event_time"))
    # Basic validity filters
    .filter(F.col("event_time").isNotNull())
    .filter(F.col("user_session").isNotNull())
    .filter((F.col("price").isNull()) | ((F.col("price") > 0) & (F.col("price") < 10000)))
    # Derived feature
    .withColumn(
        "price_tier",
        F.when(F.col("price").isNull(), "unknown")
         .when(F.col("price") < 10, "budget")
         .when(F.col("price") < 50, "mid")
         .otherwise("premium")
    )
)


In [0]:
dedup_keys = ["user_session", "event_time", "event_type", "product_id"]

silver_deduped = silver_stage.dropDuplicates(dedup_keys)


In [0]:
from delta.tables import DeltaTable

if DeltaTable.isDeltaTable(spark, SILVER_PATH):
    target = DeltaTable.forPath(spark, SILVER_PATH)
    (target.alias("t")
     .merge(
         silver_deduped.alias("s"),
         " AND ".join([f"t.{k} <=> s.{k}" for k in dedup_keys])  # null-safe equals
     )
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    (silver_deduped.write
     .format("delta")
     .mode("overwrite")
     .save(SILVER_PATH))


In [0]:
GOLD_PATH = f"{BASE_PATH}/gold/product_performance_daily"

silver = spark.read.format("delta").load(SILVER_PATH)

daily_product = (
    silver.groupBy("event_date", "product_id")
    .agg(
        F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("views_users"),
        F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("buyers_users"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price"))).alias("revenue")
    )
    .withColumn("conversion_rate_pct",
                F.when(F.col("views_users") > 0, (F.col("buyers_users") / F.col("views_users")) * 100).otherwise(F.lit(0.0)))
)

# Upsert by (event_date, product_id)
gold_keys = ["event_date", "product_id"]

if DeltaTable.isDeltaTable(spark, GOLD_PATH):
    tgt = DeltaTable.forPath(spark, GOLD_PATH)
    (tgt.alias("t")
     .merge(
         daily_product.alias("s"),
         " AND ".join([f"t.{k} = s.{k}" for k in gold_keys])
     )
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    daily_product.write.format("delta").mode("overwrite").save(GOLD_PATH)


In [0]:
GOLD_V2_PATH = f"{BASE_PATH}/gold/product_performance_daily_v2"

(daily_product
 .write.format("delta")
 .mode("overwrite")
 .partitionBy("event_date")
 .save(GOLD_V2_PATH))


In [0]:
gold = spark.read.format("delta").load(GOLD_PATH)   # if path-based
display(gold.limit(50))


In [0]:
# %python
agg_df = (
    df.filter(F.col("spend") > 50)
      .groupBy("country")
      .agg(
          F.count("*").alias("events"),
          F.round(F.avg("spend"), 2).alias("avg_spend"),
          F.max("spend").alias("max_spend")
      )
      .orderBy(F.desc("events"))
)

agg_df


In [0]:
# %python
agg_df.explain(True)


In [0]:
display(agg_df)

In [0]:
# %python
df.createOrReplaceTempView("events")


In [0]:
%sql
EXPLAIN FORMATTED
SELECT
  country,
  COUNT(*) AS events,
  ROUND(AVG(spend), 2) AS avg_spend,
  MAX(spend) AS max_spend
FROM events
WHERE spend > 50
GROUP BY country
ORDER BY events DESC;


In [0]:
%sql
SELECT
  country,
  COUNT(*) AS events,
  ROUND(AVG(spend), 2) AS avg_spend,
  MAX(spend) AS max_spend
FROM events
WHERE spend > 50
GROUP BY country
ORDER BY events DESC;


In [0]:
%fs
ls /Volumes/workspace/ecommerce/ecommerce_data/


In [0]:
df_nov.show(5, truncate=False)

In [0]:
from pyspark.sql import functions as F

filtered_df = (
    df_nov.filter(F.col("event_type").isin("purchase", "cart"))
      .filter(F.col("price") > 0)
      .select("event_time","event_type","product_id","category_code","brand","price","user_id","user_session")
)

# Materialize the cache (first action)
filtered_df.count()

# Use it multiple times (subsequent actions reuse cache)
display(
    filtered_df.groupBy("category_code")
      .agg(F.sum("price").alias("revenue"), F.count("*").alias("events"))
      .orderBy(F.desc("revenue"))
)


In [0]:
filtered_df.createOrReplaceTempView("filtered_events")

# Trigger materialization
spark.table("filtered_events").count()

In [0]:
%sql
SELECT category_code, SUM(price) AS revenue
FROM filtered_events
GROUP BY category_code
ORDER BY revenue DESC;


In [0]:
# %python
from pyspark.sql.window import Window

w = Window.partitionBy("category_code").orderBy(F.desc("revenue"))

top_brand_per_category = (
    df_nov.filter(F.col("event_type") == "purchase")
      .groupBy("category_code", "brand")
      .agg(F.sum("price").alias("revenue"))
      .withColumn("rank", F.row_number().over(w))
      .filter(F.col("rank") <= 3)
      .orderBy("category_code", "rank")
)

display(top_brand_per_category)


In [0]:
%sql
SELECT
  brand,
  ROUND(SUM(price), 2) AS total_revenue,
  COUNT(*) AS purchase_events,
  COUNT(DISTINCT user_id) AS unique_users
FROM filtered_events
WHERE event_type = 'purchase'
  AND price > 0
GROUP BY brand
ORDER BY total_revenue DESC
LIMIT 10;
